# Heatmap Plots Spearman Correlation

## Helper Functions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import spearmanr

# ── Configuration ─────────────────────────────────────────────────────────────
# Column-name prefixes that identify each statistic.
# Adjust these if your columns use different naming conventions.
STAT_PREFIXES = {
    # "mean":   "mean_",   
    "median": "median_",
    # "std":    "std_",
}

# ── extract columns for one statistic ─────────────────────────────────
def get_stat_columns(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Return a DataFrame with only columns that start with `prefix`."""
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        raise ValueError(f"No columns found starting with '{prefix}'. "
                         f"Check STAT_PREFIXES in the config section.")
    sub = df[cols].copy()
    # Strip the prefix from column names so the heatmap shows feature names only
    sub.columns = [c[len(prefix):] for c in sub.columns]
    return sub

# ── build mask that hides within-category + upper-triangle cells ──────
def build_cross_category_mask(corr: pd.DataFrame, categories: dict) -> np.ndarray:
    """
    Returns a boolean mask (True = hide cell) that hides:
      - the upper triangle (redundant with lower triangle)
      - any cell where row-feature and col-feature share the same category
    Features missing from `categories` are left ungrouped (treated as
    their own unique category, so they won't be masked against anything
    except themselves on the diagonal).
    """
    n = len(corr)
    labels = corr.columns.tolist()
    mask = np.triu(np.ones((n, n), dtype=bool))  # start with upper triangle
 
    for i, row_feat in enumerate(labels):
        for j, col_feat in enumerate(labels):
            if mask[i, j]:
                continue  # already masked by triangle
            row_cat = categories.get(row_feat, f"__unmapped_{row_feat}")
            col_cat = categories.get(col_feat, f"__unmapped_{col_feat}")

            if row_cat == col_cat:
                mask[i, j] = True
 
    return mask

# ── crop out rows/cols that are fully masked (no visible cells) ───────
def crop_to_visible(corr: pd.DataFrame, mask: np.ndarray):
    """
    Drops any row or column that is entirely masked (all True) so the
    plotted heatmap only shows axes that actually contain visible cells.
    Returns the cropped corr DataFrame and the correspondingly cropped mask.
    """
    visible = ~mask  # True where a cell WILL be shown
    keep_rows = visible.any(axis=1)   # row has at least one visible cell
    keep_cols = visible.any(axis=0)   # col has at least one visible cell
 
    cropped_corr = corr.loc[keep_rows, keep_cols]
    cropped_mask = mask[np.ix_(keep_rows, keep_cols)]
    return cropped_corr, cropped_mask

# ── plot one correlation heatmap ──────────────────────────────────────
def plot_corr_heatmap(corr: pd.DataFrame, title: str, out_path: Path, categories: dict = None) -> None:
    if categories:
        mask = build_cross_category_mask(corr, categories)
        corr, mask = crop_to_visible(corr, mask)
        fig_size = 12
    else:
        mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle only
        fig_size = max(12, len(corr))

    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    divider = make_axes_locatable(ax)
    cbar_ax = divider.append_axes("right", size="4%", pad=0.15)
     
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        linewidths=0.5,
        linecolor="white",
        square=True,
        cbar_ax=cbar_ax,
        cbar_kws={"label": "Spearman $r_s$"},
        ax=ax,
    )

    ax.set_title(title, fontsize=18, pad=25)
    ax.tick_params(axis="x", rotation=90, labelsize=14)
    ax.tick_params(axis="y", rotation=0,  labelsize=14)
    ax.tick_params(left=False, bottom=False)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")


    

FEATURE_CATEGORIES = {
    'Sleep [h]': 'sleep',
    'WASO [min]': 'sleep',
    'WASO Count': 'sleep',
    'Sleep Onset': 'sleep',
    'Sleep Midpoint': 'sleep',
    'Wakeup Time': 'sleep',
    'Light Sleep [h]': 'sleep',
    'Deep Sleep [h]': 'sleep',
    'REM Sleep [h]': 'sleep',
    'Overall Sleep Score': 'sleep',
    'SER [%]': 'sleep',
    'Steps': 'steps',
    'Sedentary Time [h]': 'steps',
    'Movement Time [h]': 'steps',
    'Movement/Sedentary Ratio': 'steps',
    'Steps 2h after Wakeup': 'steps',
    'Steps 4h after Wakeup': 'steps',
    'Steps 2h before Onset': 'steps',
    'Steps 4h before Onset': 'steps',
    'HR [bpm]': 'hr',
    'HRV (RMSSD) [ms]': 'hr',
    'Nocturnal HR [bpm]': 'hr',
    'Nocturnal HRV (RMSSD) [ms]': 'hr',
}


SLEEP_LABEL_MAP = {
    'median_sleep_duration':'Sleep [h]',
    'median_waso': 'WASO [min]',
    'median_awake': 'WASO Count',
    'median_sleep_onset_s_adj': 'Sleep Onset',
    'median_midpoint_s_adj': 'Sleep Midpoint',
    'median_wakeup_seconds': 'Wakeup Time',
    'median_light_sleep_duration': 'Light Sleep [h]',
    'median_deep_sleep_duration': 'Deep Sleep [h]',
    'median_rem_sleep_duration': 'REM Sleep [h]',
    'median_overall_sleep_score':'Overall Sleep Score',
    'median_ser': 'SER [%]',

}

STEP_LABEL_MAP = {
    'median_steps': 'Steps',
    'median_sedentary_time_h': 'Sedentary Time [h]',
    'median_movement_time_h': 'Movement Time [h]',
    'median_ratio': 'Movement/Sedentary Ratio',
    'median_steps_2h': 'Steps 2h after Wakeup',
    'median_steps_4h': 'Steps 4h after Wakeup',
    'median_steps_onset_2h': 'Steps 2h before Onset',
    'median_steps_onset_4h': 'Steps 4h before Onset',
}

HR_LABEL_MAP = {
    'mean_hr': 'HR [bpm]',
    'mean_rmssd': 'HRV (RMSSD) [ms]',
    'mean_hr_nocturnal': 'Nocturnal HR [bpm]',
    'mean_rmssd_nocturnal': 'Nocturnal HRV (RMSSD) [ms]',
}

ANTHROPOMETRIC_LABEL_MAP = {
    'height_m': 'Height [m]',
    'weight_kg': 'Weight [kg]',
    'waist_circ_cm': 'Waist Circumference [cm]',
    'bmi': 'BMI [kg/m²]',
    'age_at_visit': 'Age [y]',
}
corr_dict = {}

## Sleep Features

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv("../../output/1_feature_extraction/df_features_sleep_2026-07-08.csv")
display(df.columns)
df_sleep_stages_stats = pd.read_csv('../../output/1_feature_extraction/df_features_sleep_stages_2026-07-08.csv')
display(df_sleep_stages_stats.columns)

df = df.merge(df_sleep_stages_stats, on='study_id', how='outer')
print(f"Loaded {df.shape[0]} participants × {df.shape[1]} columns")

# Convert time columns to seconds
time_columns = ['median_sleep_onset', 'median_midpoint', 'median_wakeup', 'mean_sleep_onset', 'mean_midpoint', 'mean_wakeup']

for col in time_columns:
    df[col + '_seconds'] = pd.to_datetime(df[col], format='mixed').dt.time.apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )

# Midnight crossover adjustment
df['median_sleep_onset_s_adj'] = df['median_sleep_onset_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
df['median_midpoint_s_adj'] = df['median_midpoint_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)

df['mean_sleep_onset_s_adj'] = df['mean_sleep_onset_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
df['mean_midpoint_s_adj'] = df['mean_midpoint_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)

#drop seconds time columns
df = df.drop(columns=['median_sleep_onset_seconds', 'median_midpoint_seconds', 'mean_sleep_onset_seconds', 'mean_midpoint_seconds'])
# df = df.rename(columns=lambda x: x.replace('_seconds_adjusted', ''))  # Remove '_seconds_adjusted' suffix
display(df)
display(df.columns)

df_sleep = df[['study_id', 
         'median_sleep_duration', 
         'median_waso', 
         'median_awake', 
         'median_sleep_onset_s_adj', 
         'median_midpoint_s_adj', 
         'median_wakeup_seconds', 
         'median_light_sleep_duration', 
         'median_deep_sleep_duration', 
         'median_rem_sleep_duration', 
         'median_overall_sleep_score',
         'median_ser', ]].copy()

# ── Main: one heatmap per statistic ───────────────────────────────────────────


# Drop non-numeric columns just in case
stat_df_sleep = df_sleep.select_dtypes(include="number")


stat_df_sleep = stat_df_sleep.rename(columns=SLEEP_LABEL_MAP)

rho, p = spearmanr(stat_df_sleep, nan_policy="omit")

corr = pd.DataFrame(
    rho,
    index=stat_df_sleep.columns,
    columns=stat_df_sleep.columns
)

p_values = pd.DataFrame(
    p,
    index=stat_df_sleep.columns,
    columns=stat_df_sleep.columns
)


display(stat_df_sleep.head())
corr_dict["Sleep"] = corr

date = datetime.now().strftime("%Y-%m-%d")
p_values.to_csv(Path(f"../../output/2_2_correlations/sleep_feature_correlation_pvals_{date}.csv"), index=True)

title = f"Sleep Feature Spearman Correlation Matrix (n={len(stat_df_sleep)})"
plot_corr_heatmap(corr, title, out_path=Path(f"../../plots/Correlation/heatmap_sleep_spearman_{date}.png"))

## Activity Feature

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df_steps = pd.read_csv("../../output/1_feature_extraction/df_features_step_2026-07-08.csv")
display(df_steps.columns)
#!exclude DEC_46 from all step features
df_steps = df_steps[df_steps['study_id'] != 'DEC_46']
# df_steps = df_steps[df_steps['study_id'] != 'DEC_39']

print(f"Loaded {df_steps.shape[0]} participants × {df_steps.shape[1]} columns")


df_steps_crop = df_steps[['study_id', 
                    'median_steps', 
                    'median_sedentary_time_h', 
                    'median_movement_time_h', 
                    'median_ratio', 
                    'median_steps_2h', 
                    'median_steps_4h', 
                    'median_steps_onset_2h', 
                    'median_steps_onset_4h']].copy()



# Drop non-numeric columns just in case
stat_df_steps = df_steps_crop.select_dtypes(include="number")

stat_df_steps = stat_df_steps.rename(columns=STEP_LABEL_MAP)
rho, p = spearmanr(stat_df_steps, nan_policy="omit")

corr = pd.DataFrame(
    rho,
    index=stat_df_steps.columns,
    columns=stat_df_steps.columns
)

p_values = pd.DataFrame(
    p,
    index=stat_df_steps.columns,
    columns=stat_df_steps.columns
)


display(stat_df_steps.head())
corr_dict["Steps"] = corr

date = datetime.now().strftime("%Y-%m-%d")
p_values.to_csv(Path(f"../../output/2_2_correlations/step_feature_correlation_pvals_{date}.csv"), index=True)

title = f"Activity Features Spearman Correlation Matrix (n={len(stat_df_steps)})"
plot_corr_heatmap(corr, title, out_path=Path(f"../../plots/Correlation/heatmap_steps_spearman_{date}.png"))

## HR/HRV

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df_hr = pd.read_csv("../../output/1_feature_extraction/df_features_hr_2026-07-08.csv")
df_nocturnal = pd.read_csv("../../output/1_feature_extraction/df_features_nocturnal_hr_2026-07-08.csv")
#exclude patients with less than 7 days/nights of data
df_hr = df_hr[df_hr['n_days'] >= 7]
df_nocturnal = df_nocturnal[df_nocturnal['n_nights'] >= 7]

df_hr = df_hr[['mean_hr', 'mean_rmssd', 'n_days', 'study_id']].copy()
df_nocturnal = df_nocturnal[['mean_hr', 'mean_rmssd', 'n_nights', 'study_id']].copy()

df_hr = df_hr.merge(df_nocturnal, on='study_id', how='outer', suffixes=('', '_nocturnal'))
display(df_hr.columns)

print(f"Loaded {df_hr.shape[0]} participants × {df_hr.shape[1]} columns")

df_hr_crop = df_hr[['study_id', 'mean_hr', 'mean_rmssd', 'mean_hr_nocturnal', 'mean_rmssd_nocturnal']].copy()

# Drop non-numeric columns just in case
stat_df_hr = df_hr_crop.select_dtypes(include="number")

stat_df_hr = stat_df_hr.rename(columns=HR_LABEL_MAP)

rho, p = spearmanr(stat_df_hr, nan_policy="omit")

corr = pd.DataFrame(
    rho,
    index=stat_df_hr.columns,
    columns=stat_df_hr.columns
)

p_values = pd.DataFrame(
    p,
    index=stat_df_hr.columns,
    columns=stat_df_hr.columns
)

display(stat_df_hr.head())
corr_dict["HR/HRV"] = corr

date = datetime.now().strftime("%Y-%m-%d")
p_values.to_csv(Path(f"../../output/2_2_correlations/hr_feature_correlation_pvals_{date}.csv"), index=True)

title = f"HR/HRV Feature Spearman Correlation Matrix (n={len(stat_df_hr)})"
plot_corr_heatmap(corr, title, out_path=Path(f"../../plots/Correlation/heatmap_hr_spearman_{date}.png"))

## Activity & Sleep Features

In [ ]:
df_sleep_steps = df_sleep.merge(df_steps_crop, on='study_id', how='outer')


display(df_sleep_steps.columns)

print(f"Loaded {df_sleep_steps.shape[0]} participants × {df_sleep_steps.shape[1]} columns")

sleep_step_stat_df = df_sleep_steps.copy()
# Drop non-numeric columns just in case
sleep_step_stat_df = sleep_step_stat_df.select_dtypes(include="number")

sleep_step_stat_df = sleep_step_stat_df.rename(columns={**SLEEP_LABEL_MAP, **STEP_LABEL_MAP})


rho, p = spearmanr(sleep_step_stat_df, nan_policy="omit")

corr = pd.DataFrame(
    rho,
    index=sleep_step_stat_df.columns,
    columns=sleep_step_stat_df.columns
)

p_values = pd.DataFrame(
    p,
    index=sleep_step_stat_df.columns,
    columns=sleep_step_stat_df.columns
)


display(sleep_step_stat_df.head())

date = datetime.now().strftime("%Y-%m-%d")
p_values.to_csv(Path(f"../../output/2_2_correlations/sleep_step_feature_correlation_pvals_{date}.csv"), index=True)

title= f"Sleep and Activity Features Spearman Correlation Matrix"
plot_corr_heatmap(corr, title, out_path=Path(f"../../plots/Correlation/heatmap_sleep_steps_spearman_{date}.png"), categories=FEATURE_CATEGORIES)

## Sleep & HR

In [ ]:
df_sleep_hr = df_sleep.merge(df_hr_crop, on='study_id', how='outer')


display(df_sleep_hr.columns)

print(f"Loaded {df_sleep_hr.shape[0]} participants × {df_sleep_hr.shape[1]} columns")

sleep_hr_stat_df = df_sleep_hr.copy()
# Drop non-numeric columns just in case
sleep_hr_stat_df = sleep_hr_stat_df.select_dtypes(include="number")

sleep_hr_stat_df = sleep_hr_stat_df.rename(columns={**SLEEP_LABEL_MAP, **HR_LABEL_MAP})


rho, p = spearmanr(sleep_hr_stat_df, nan_policy="omit")

corr = pd.DataFrame(
    rho,
    index=sleep_hr_stat_df.columns,
    columns=sleep_hr_stat_df.columns
)

p_values = pd.DataFrame(
    p,
    index=sleep_hr_stat_df.columns,
    columns=sleep_hr_stat_df.columns
)


display(sleep_hr_stat_df.head())

date = datetime.now().strftime("%Y-%m-%d")
p_values.to_csv(Path(f"../../output/2_2_correlations/sleep_hr_feature_correlation_pvals_{date}.csv"), index=True)

title= f"Sleep and HR Features Spearman Correlation Matrix"
plot_corr_heatmap(corr, title,out_path=Path(f"../../plots/Correlation/heatmap_sleep_hr_spearman_{date}.png"), categories=FEATURE_CATEGORIES)

## Activity & HR

In [ ]:
df_step_hr = df_steps_crop.merge(df_hr_crop, on='study_id', how='outer')


display(df_step_hr.columns)

print(f"Loaded {df_step_hr.shape[0]} participants × {df_step_hr.shape[1]} columns")

step_hr_stat_df = df_step_hr.copy()
# Drop non-numeric columns just in case
step_hr_stat_df = step_hr_stat_df.select_dtypes(include="number")

step_hr_stat_df = step_hr_stat_df.rename(columns={**STEP_LABEL_MAP, **HR_LABEL_MAP})

rho, p = spearmanr(step_hr_stat_df, nan_policy="omit")

corr = pd.DataFrame(
    rho,
    index=step_hr_stat_df.columns,
    columns=step_hr_stat_df.columns
)

p_values = pd.DataFrame(
    p,
    index=step_hr_stat_df.columns,
    columns=step_hr_stat_df.columns
)

display(step_hr_stat_df.head())

date = datetime.now().strftime("%Y-%m-%d")
p_values.to_csv(Path(f"../../output/2_2_correlations/step_hr_feature_correlation_pvals_{date}.csv"), index=True)

title= f"Activity and HR Features Spearman Correlation Matrix"
plot_corr_heatmap(corr, title, out_path=Path(f"../../plots/Correlation/heatmap_step_hr_spearman_{date}.png"), categories=FEATURE_CATEGORIES)

## All

In [ ]:
stat_df = df_sleep.merge(df_steps_crop, on='study_id', how='outer')
stat_df = stat_df.merge(df_hr_crop, on='study_id', how='outer')

display(stat_df.columns)

print(f"Loaded {stat_df.shape[0]} participants × {stat_df.shape[1]} columns")

stat_df = stat_df.copy()
# Drop non-numeric columns just in case
stat_df = stat_df.select_dtypes(include="number")
stat_df = stat_df.rename(columns={**SLEEP_LABEL_MAP, **HR_LABEL_MAP, **STEP_LABEL_MAP})

rho, p = spearmanr(stat_df, nan_policy="omit")

corr = pd.DataFrame(
    rho,
    index=stat_df.columns,
    columns=stat_df.columns
)

p_values = pd.DataFrame(
    p,
    index=stat_df.columns,
    columns=stat_df.columns
)


display(stat_df.head())

date = datetime.now().strftime("%Y-%m-%d")
p_values.to_csv(Path(f"../../output/2_2_correlations/all_feature_correlation_pvals_{date}.csv"), index=True)

title= f"Sleep, Activity and HR/HRV Features Spearman Correlation Matrix"
plot_corr_heatmap(corr, title, out_path=Path(f"../../plots/Correlation/heatmap_all_spearman_{date}.png"))